In [1]:
# Cell 1: environment configuration
import importlib.util
import os
import random
import subprocess
import sys
from datetime import datetime
from pathlib import Path

def _pip_install(packages):
    if packages:
        print("Installing missing packages:", packages)
        subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])

required = {
    "numpy": "numpy",
    "h5py": "h5py",
    "sklearn": "scikit-learn",
    "tqdm": "tqdm",
    "tensorboard": "tensorboard",
}
missing = [pip_name for module_name, pip_name in required.items() if importlib.util.find_spec(module_name) is None]
_pip_install(missing)

if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch is not installed. Install a CUDA build matching this machine first.")

import h5py
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm

def _has_data_layout(root: Path) -> bool:
    return (
        (root / "BCIC2A" / "train.h5").exists()
        and (root / "BCIC2A" / "val.h5").exists()
        and (root / "BCIC2A" / "test_x_only.h5").exists()
        # and (root / "suppplementary_BCIC2A_cleaned" / "train.backup_before_csv_20260517_175342.h5").exists()
        # and (root / "suppplementary_BCIC2A_cleaned" / "val.backup_before_csv_20260517_175342.h5").exists()
        and (root / "suppplementary_BCIC2A_cleaned" / "train.h5").exists()
        and (root / "suppplementary_BCIC2A_cleaned" / "val.h5").exists()
    )

def locate_data_root() -> Path:
    env_root = os.environ.get("DATA_ROOT") or os.environ.get("PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).expanduser().resolve())
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend([p.resolve() for p in cwd.iterdir() if p.is_dir()])
    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if _has_data_layout(candidate):
            return candidate
    raise FileNotFoundError("Cannot locate BCIC2A and suppplementary_BCIC2A_cleaned data folders.")

DATA_ROOT = locate_data_root()
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", str(DATA_ROOT))).expanduser().resolve()
CODE_DIR = PROJECT_ROOT / "Codes" / "BCIC2A"
BCI_DIR = DATA_ROOT / "BCIC2A"
SUPP_DIR = DATA_ROOT / "suppplementary_BCIC2A_cleaned"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("DATA_ROOT:", DATA_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("CODE_DIR:", CODE_DIR)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS


DATA_ROOT: /root/BCI
PROJECT_ROOT: /root/BCI
CODE_DIR: /root/BCI/Codes/BCIC2A
Device: cuda
GPU: NVIDIA GeForce RTX 4090 D


# BCIC2A FusionEEGNet Training

This notebook trains a supervised multi-scale FusionEEGNet classifier directly on the integrated supplementary BCIC2A data. It does not run masked pretraining.


In [2]:
# Cell 2: training configuration
RUN_MODE = os.environ.get("RUN_MODE", "terminal_train").lower()
if RUN_MODE not in {"local_test", "terminal_train"}:
    raise ValueError("RUN_MODE must be 'local_test' or 'terminal_train'")

IS_FULL_TRAINING = RUN_MODE == "terminal_train"

SEED = int(os.environ.get("SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_NAME = f"bcic2a_fusion_eegnet_{RUN_TAG}"
TB_ROOT = CODE_DIR / "runs" / "bcic2a_fusion_eegnet" / RUN_TAG
CKPT_ROOT = CODE_DIR / "checkpoints" / "bcic2a_fusion_eegnet" / RUN_TAG
TB_ROOT.mkdir(parents=True, exist_ok=True)
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

if IS_FULL_TRAINING:
    EPOCHS = 260
    BATCH_SIZE = 128
    NUM_WORKERS = min(8, os.cpu_count() or 0)
    MAX_TRAIN_SAMPLES = None
    MAX_VAL_SAMPLES = None
    MAX_TEST_SAMPLES = None
    EXPORT_TEST_PREDICTIONS = True
else:
    EPOCHS = 2
    BATCH_SIZE = 32
    NUM_WORKERS = 0
    MAX_TRAIN_SAMPLES = 512
    MAX_VAL_SAMPLES = 256
    MAX_TEST_SAMPLES = 64
    EXPORT_TEST_PREDICTIONS = False

LR = float(os.environ.get("LR", "5e-4"))
WEIGHT_DECAY = float(os.environ.get("WEIGHT_DECAY", "2e-4"))
DROPOUT = float(os.environ.get("DROPOUT", "0.40"))
LABEL_SMOOTHING = float(os.environ.get("LABEL_SMOOTHING", "0.0"))
GRAD_CLIP_NORM = 1.0
EARLY_STOP_PATIENCE = 70

PIN_MEMORY = torch.cuda.is_available()
PERSISTENT_WORKERS = NUM_WORKERS > 0
USE_AMP = torch.cuda.is_available()

# Lightweight EEG augmentation. Kept off during evaluation.
USE_AUGMENT = True
AUG_NOISE_STD = 0.003
AUG_SCALE_MIN = 0.90
AUG_SCALE_MAX = 1.10
AUG_TIME_SHIFT = 40
CHANNEL_DROPOUT_PROB = 0.15
MAX_DROPPED_CHANNELS = 2

print("RUN_MODE:", RUN_MODE)
print("RUN_NAME:", RUN_NAME)
print(f"EPOCHS={EPOCHS}, BATCH_SIZE={BATCH_SIZE}, NUM_WORKERS={NUM_WORKERS}")
print(f"LR={LR}, WEIGHT_DECAY={WEIGHT_DECAY}, DROPOUT={DROPOUT}, LABEL_SMOOTHING={LABEL_SMOOTHING}, AMP={USE_AMP}")


RUN_MODE: terminal_train
RUN_NAME: bcic2a_fusion_eegnet_20260517_191949
EPOCHS=260, BATCH_SIZE=128, NUM_WORKERS=8
LR=0.0005, WEIGHT_DECAY=0.0002, DROPOUT=0.4, LABEL_SMOOTHING=0.0, AMP=True


In [3]:
# Cell 3: datasets and dataloaders
class TrainH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = Path(h5_path)
        with h5py.File(self.h5_path, "r") as f:
            self.x = torch.tensor(f["X"][()], dtype=torch.float32)
            self.y = torch.tensor(f["y"][()], dtype=torch.long)
            self.attrs = dict(f.attrs)
        if len(self.x) != len(self.y):
            raise ValueError(f"X/y length mismatch in {self.h5_path}")

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

class XOnlyH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = Path(h5_path)
        with h5py.File(self.h5_path, "r") as f:
            self.x = torch.tensor(f["X"][()], dtype=torch.float32)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx]

def maybe_subset(ds, max_items, name):
    if max_items is None or len(ds) <= max_items:
        return ds
    n = max(1, min(len(ds), int(max_items)))
    print(f"Subset {name}: {n}/{len(ds)}")
    return Subset(ds, list(range(n)))

def seed_worker(worker_id):
    worker_seed = (SEED + worker_id) % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

official_train_ds = TrainH5Dataset(BCI_DIR / "train.h5")
official_val_ds = TrainH5Dataset(BCI_DIR / "val.h5")
# supp_train_ds = TrainH5Dataset(SUPP_DIR / "train.backup_before_csv_20260517_175342.h5")
# supp_val_ds = TrainH5Dataset(SUPP_DIR / "val.backup_before_csv_20260517_175342.h5")
supp_train_ds = TrainH5Dataset(SUPP_DIR / "train.h5")
supp_val_ds = TrainH5Dataset(SUPP_DIR / "val.h5")
test_full_ds = XOnlyH5Dataset(BCI_DIR / "test_x_only.h5")

# Supervised training uses the integrated supplementary data.
# Official train+val stay held out as the clean validation set.
train_full_ds = ConcatDataset([supp_train_ds, supp_val_ds])
val_full_ds = ConcatDataset([official_train_ds, official_val_ds])
train_ds = maybe_subset(train_full_ds, MAX_TRAIN_SAMPLES, "train")
val_ds = maybe_subset(val_full_ds, MAX_VAL_SAMPLES, "val")
test_ds = maybe_subset(test_full_ds, MAX_TEST_SAMPLES, "test")

x0, _ = supp_train_ds[0]
CHANS, TIME_POINTS = x0.shape
NUM_CLASSES = int(
    max(
        official_train_ds.y.max(),
        official_val_ds.y.max(),
        supp_train_ds.y.max(),
        supp_val_ds.y.max(),
    ).item()
) + 1

g = torch.Generator()
g.manual_seed(SEED)
loader_kwargs = dict(
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    worker_init_fn=seed_worker,
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=g, **loader_kwargs)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

def label_counts(ds):
    parts = []
    if isinstance(ds, ConcatDataset):
        for child in ds.datasets:
            parts.append(child.y)
        y = torch.cat(parts)
    elif isinstance(ds, Subset):
        base = ds.dataset
        if isinstance(base, ConcatDataset):
            labels = []
            for idx in ds.indices:
                labels.append(base[idx][1])
            y = torch.tensor(labels)
        else:
            y = base.y[ds.indices]
    else:
        y = ds.y
    return torch.bincount(y, minlength=NUM_CLASSES).tolist()

print(f"Supp train:       {len(supp_train_ds)}")
print(f"Supp val:         {len(supp_val_ds)}")
print(f"Final train size: {len(train_ds)} / full {len(train_full_ds)}")
print(f"Official train:   {len(official_train_ds)}")
print(f"Official val:     {len(official_val_ds)}")
print(f"Final val size:   {len(val_ds)} / full {len(val_full_ds)}")
print(f"Test size:        {len(test_ds)} / full {len(test_full_ds)}")
print(f"Input shape:      C={CHANS}, T={TIME_POINTS}, classes={NUM_CLASSES}")
print("Train label counts:", label_counts(train_full_ds))
print("Val label counts:  ", label_counts(val_full_ds))


Supp train:       5009
Supp val:         2503
Final train size: 7512 / full 7512
Official train:   720
Official val:     360
Final val size:   1080 / full 1080
Test size:        360 / full 360
Input shape:      C=22, T=800, classes=4
Train label counts: [1868, 1887, 1870, 1887]
Val label counts:   [270, 270, 270, 270]


In [4]:
# Cell 4: FusionEEGNet model
class SqueezeExcite(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        hidden = max(4, channels // reduction)
        self.net = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(channels, hidden),
            nn.ELU(),
            nn.Linear(hidden, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        scale = self.net(x).view(x.size(0), x.size(1), 1, 1)
        return x * scale

class TemporalBranch(nn.Module):
    def __init__(self, out_channels, kernel_length, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, out_channels, kernel_size=(1, kernel_length), padding=(0, kernel_length // 2), bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ELU(),
            nn.Dropout(dropout * 0.5),
        )

    def forward(self, x):
        return self.net(x)

class FusionEEGNet(nn.Module):
    def __init__(
        self,
        chans,
        time_points,
        num_classes,
        branch_channels=8,
        temporal_kernels=(32, 64, 128),
        depth_multiplier=2,
        sep_kernel_length=16,
        pool1=4,
        pool2=8,
        dropout=0.40,
    ):
        super().__init__()
        temporal_channels = branch_channels * len(temporal_kernels)
        spatial_channels = temporal_channels * depth_multiplier

        self.temporal_branches = nn.ModuleList(
            [TemporalBranch(branch_channels, k, dropout) for k in temporal_kernels]
        )
        self.temporal_fuse = nn.Sequential(
            nn.Conv2d(temporal_channels, temporal_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(temporal_channels),
            nn.ELU(),
        )
        self.spatial = nn.Sequential(
            nn.Conv2d(temporal_channels, spatial_channels, kernel_size=(chans, 1), groups=temporal_channels, bias=False),
            nn.BatchNorm2d(spatial_channels),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, pool1), stride=(1, pool1)),
            nn.Dropout(dropout),
        )
        self.se = SqueezeExcite(spatial_channels)
        self.separable = nn.Sequential(
            nn.Conv2d(spatial_channels, spatial_channels, kernel_size=(1, sep_kernel_length), padding=(0, sep_kernel_length // 2), groups=spatial_channels, bias=False),
            nn.Conv2d(spatial_channels, spatial_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(spatial_channels),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, pool2), stride=(1, pool2)),
            nn.Dropout(dropout),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, 1, chans, time_points)
            emb_dim = self._forward_features(dummy).flatten(1).shape[1]
        self.classifier = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Dropout(dropout),
            nn.Linear(emb_dim, max(64, emb_dim // 4)),
            nn.ELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(max(64, emb_dim // 4), num_classes),
        )

    def _forward_features(self, x):
        x = torch.cat([branch(x) for branch in self.temporal_branches], dim=1)
        x = self.temporal_fuse(x)
        x = self.spatial(x)
        x = self.se(x)
        x = self.separable(x)
        return x

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self._forward_features(x)
        return self.classifier(x.flatten(1))

model_probe = FusionEEGNet(CHANS, TIME_POINTS, NUM_CLASSES, dropout=DROPOUT)
print(model_probe)
print(f"Parameters: {sum(p.numel() for p in model_probe.parameters()):,}")


FusionEEGNet(
  (temporal_branches): ModuleList(
    (0): TemporalBranch(
      (net): Sequential(
        (0): Conv2d(1, 8, kernel_size=(1, 32), stride=(1, 1), padding=(0, 16), bias=False)
        (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ELU(alpha=1.0)
        (3): Dropout(p=0.2, inplace=False)
      )
    )
    (1): TemporalBranch(
      (net): Sequential(
        (0): Conv2d(1, 8, kernel_size=(1, 64), stride=(1, 1), padding=(0, 32), bias=False)
        (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ELU(alpha=1.0)
        (3): Dropout(p=0.2, inplace=False)
      )
    )
    (2): TemporalBranch(
      (net): Sequential(
        (0): Conv2d(1, 8, kernel_size=(1, 128), stride=(1, 1), padding=(0, 64), bias=False)
        (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ELU(alpha=1.0)
        (3): Dropout(p=0.2, inplace=False)
      )
  

In [5]:
# Cell 5: training and evaluation helpers
def augment_batch(x):
    if not USE_AUGMENT:
        return x
    if AUG_TIME_SHIFT > 0:
        shifts = torch.randint(-AUG_TIME_SHIFT, AUG_TIME_SHIFT + 1, (x.size(0),), device=x.device)
        x = torch.stack([torch.roll(xi, int(shift.item()), dims=-1) for xi, shift in zip(x, shifts)], dim=0)
    if AUG_SCALE_MAX > 0:
        scale = torch.empty((x.size(0), 1, 1), device=x.device).uniform_(AUG_SCALE_MIN, AUG_SCALE_MAX)
        x = x * scale
    if AUG_NOISE_STD > 0:
        x = x + torch.randn_like(x) * AUG_NOISE_STD
    if CHANNEL_DROPOUT_PROB > 0 and MAX_DROPPED_CHANNELS > 0:
        for i in range(x.size(0)):
            if torch.rand((), device=x.device) < CHANNEL_DROPOUT_PROB:
                n_drop = int(torch.randint(1, MAX_DROPPED_CHANNELS + 1, (), device=x.device).item())
                ch = torch.randperm(x.size(1), device=x.device)[:n_drop]
                x[i, ch, :] = 0
    return x

def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    logits_parts = []
    label_parts = []

    with torch.set_grad_enabled(is_train):
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            if is_train:
                xb = augment_batch(xb)
                optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(xb)
                loss = criterion(logits, yb)
            if is_train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()

            bs = yb.size(0)
            total_loss += loss.item() * bs
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += bs
            if not is_train:
                logits_parts.append(logits.detach().cpu())
                label_parts.append(yb.detach().cpu())

    metrics = {
        "loss": total_loss / total_count,
        "acc": total_correct / total_count,
    }
    if not is_train:
        metrics["logits"] = torch.cat(logits_parts)
        metrics["labels"] = torch.cat(label_parts)
        pred = metrics["logits"].argmax(dim=1).numpy()
        labels = metrics["labels"].numpy()
        metrics["balanced_acc"] = balanced_accuracy_score(labels, pred)
        metrics["preds"] = pred
    return metrics

@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    parts = []
    for xb in tqdm(loader, desc="Predict", leave=False):
        xb = xb.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            parts.append(model(xb).detach().cpu())
    return torch.cat(parts)


In [6]:
# Cell 6: train FusionEEGNet
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
model = FusionEEGNet(CHANS, TIME_POINTS, NUM_CLASSES, dropout=DROPOUT).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
writer = SummaryWriter(log_dir=str(TB_ROOT))

best_val_acc = -1.0
best_epoch = 0
no_improve = 0
best_path = CKPT_ROOT / "best.pt"

for epoch in range(1, EPOCHS + 1):
    train_m = run_epoch(model, train_loader, criterion, optimizer=optimizer, scaler=scaler)
    val_m = run_epoch(model, val_loader, criterion, optimizer=None, scaler=None)
    scheduler.step()

    writer.add_scalar("loss/train", train_m["loss"], epoch)
    writer.add_scalar("loss/val", val_m["loss"], epoch)
    writer.add_scalar("acc/train", train_m["acc"], epoch)
    writer.add_scalar("acc/val", val_m["acc"], epoch)
    writer.add_scalar("balanced_acc/val", val_m["balanced_acc"], epoch)
    writer.add_scalar("lr", optimizer.param_groups[0]["lr"], epoch)

    print(
        f"[FusionEEGNet] Epoch {epoch:03d}/{EPOCHS} | "
        f"Train Loss {train_m['loss']:.4f} Acc {train_m['acc']:.4f} | "
        f"Val Loss {val_m['loss']:.4f} Acc {val_m['acc']:.4f} BalAcc {val_m['balanced_acc']:.4f}"
    )

    payload = {
        "epoch": epoch,
        "model_name": "FusionEEGNet",
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "val_acc": val_m["acc"],
        "val_balanced_acc": val_m["balanced_acc"],
        "train_acc": train_m["acc"],
        "chans": CHANS,
        "time_points": TIME_POINTS,
        "num_classes": NUM_CLASSES,
        "run_tag": RUN_TAG,
        "run_mode": RUN_MODE,
    }
    torch.save(payload, CKPT_ROOT / "last.pt")
    if val_m["acc"] > best_val_acc:
        best_val_acc = val_m["acc"]
        best_epoch = epoch
        no_improve = 0
        torch.save(payload, best_path)
    else:
        no_improve += 1
        if IS_FULL_TRAINING and no_improve >= EARLY_STOP_PATIENCE:
            print(f"Early stopping at epoch {epoch}; best epoch {best_epoch}")
            break

writer.close()
print(f"Best val acc: {best_val_acc:.4f} @ epoch {best_epoch}")
print("Best checkpoint:", best_path)


[FusionEEGNet] Epoch 001/260 | Train Loss 1.3802 Acc 0.3272 | Val Loss 1.5398 Acc 0.3250 BalAcc 0.3250
[FusionEEGNet] Epoch 002/260 | Train Loss 1.2849 Acc 0.4176 | Val Loss 1.1355 Acc 0.4944 BalAcc 0.4944
[FusionEEGNet] Epoch 003/260 | Train Loss 1.2194 Acc 0.4570 | Val Loss 1.0784 Acc 0.5361 BalAcc 0.5361
[FusionEEGNet] Epoch 004/260 | Train Loss 1.1794 Acc 0.4798 | Val Loss 1.0395 Acc 0.5639 BalAcc 0.5639
[FusionEEGNet] Epoch 005/260 | Train Loss 1.1530 Acc 0.4953 | Val Loss 1.0063 Acc 0.5648 BalAcc 0.5648
[FusionEEGNet] Epoch 006/260 | Train Loss 1.1213 Acc 0.5189 | Val Loss 0.9745 Acc 0.6111 BalAcc 0.6111
[FusionEEGNet] Epoch 007/260 | Train Loss 1.1066 Acc 0.5346 | Val Loss 0.9342 Acc 0.6120 BalAcc 0.6120
[FusionEEGNet] Epoch 008/260 | Train Loss 1.0800 Acc 0.5383 | Val Loss 0.9011 Acc 0.6389 BalAcc 0.6389
[FusionEEGNet] Epoch 009/260 | Train Loss 1.0589 Acc 0.5491 | Val Loss 0.8752 Acc 0.6491 BalAcc 0.6491
[FusionEEGNet] Epoch 010/260 | Train Loss 1.0410 Acc 0.5583 | Val Loss 0.

In [7]:
# Cell 7: final validation report
best_payload = torch.load(best_path, map_location=device)
model.load_state_dict(best_payload["model_state"])
final_val = run_epoch(model, val_loader, criterion, optimizer=None, scaler=None)
labels = final_val["labels"].numpy()
preds = final_val["preds"]

print(f"Final val acc: {accuracy_score(labels, preds):.4f}")
print(f"Final val balanced acc: {balanced_accuracy_score(labels, preds):.4f}")
print("Confusion matrix:\n", confusion_matrix(labels, preds))
print(classification_report(labels, preds, digits=4))


Final val acc: 0.9889
Final val balanced acc: 0.9889
Confusion matrix:
 [[267   1   2   0]
 [  2 266   0   2]
 [  2   0 266   2]
 [  1   0   0 269]]
              precision    recall  f1-score   support

           0     0.9816    0.9889    0.9852       270
           1     0.9963    0.9852    0.9907       270
           2     0.9925    0.9852    0.9888       270
           3     0.9853    0.9963    0.9908       270

    accuracy                         0.9889      1080
   macro avg     0.9889    0.9889    0.9889      1080
weighted avg     0.9889    0.9889    0.9889      1080



In [8]:
# Cell 8: optional ordered test prediction export
out_path = BCI_DIR / "BCIC2A_fusion_eegnet.txt"
if EXPORT_TEST_PREDICTIONS:
    test_logits = predict_logits(model, test_loader)
    test_pred = test_logits.argmax(dim=1).numpy().astype(int).tolist()
    if len(test_pred) != len(test_ds):
        raise RuntimeError(f"Prediction count mismatch: {len(test_pred)} != {len(test_ds)}")
    with open(out_path, "w", encoding="utf-8") as f:
        for pred in test_pred:
            f.write(f"{pred}\n")
    print(f"Saved {len(test_pred)} ordered labels to {out_path}")
else:
    print("EXPORT_TEST_PREDICTIONS=False. Use RUN_MODE=terminal_train to write test predictions.")


Predict:   0%|          | 0/3 [00:00<?, ?it/s]

Saved 360 ordered labels to /root/BCI/BCIC2A/BCIC2A_fusion_eegnet.txt


## Terminal Training

From this `Codes/BCIC2A` folder in PowerShell:

```powershell
$env:RUN_MODE = "terminal_train"
& "d:\Tools\miniconda\envs\ML\python.exe" -m jupyter nbconvert --to notebook --execute "BCIC2A_FusionEEGNet_train_local.ipynb" --output "BCIC2A_FusionEEGNet_train_runned.ipynb"
```

TensorBoard:

```powershell
& "d:\Tools\miniconda\envs\ML\Scripts\tensorboard.exe" --logdir "runs\bcic2a_fusion_eegnet" --host 0.0.0.0 --port 6006
```
